In [1]:
!pip install mysql-connector-python

     ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/16.4 MB 2.0 MB/s eta 0:00:08
     ---------------------------------------- 0.1/16.4 MB 1.1 MB/s eta 0:00:16
     ---------------------------------------- 0.1/16.4 MB 1.1 MB/s eta 0:00:16
      --------------------------------------- 0.3/16.4 MB 1.5 MB/s eta 0:00:11
     - -------------------------------------- 0.5/16.4 MB 2.1 MB/s eta 0:00:08
     - -------------------------------------- 0.6/16.4 MB 2.4 MB/s eta 0:00:07
     - -------------------------------------- 0.6/16.4 MB 2.4 MB/s eta 0:00:07
     -- ------------------------------------- 0.8/16.4 MB 2.4 MB/s eta 0:00:07
     -- ------------------------------------- 1.1/16.4 MB 2.7 MB/s eta 0:00:06
     -- ------------------------------------- 1.2/16.4 MB 2.7 MB/s eta 0:00:06
     --- ------------------------------------ 1.3/16.4 MB 2.8 MB/s eta 0:00:06
     --- ------------------------------------ 1.3/16.4 MB 2


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456",
    database="company_db"
)

# Create a cursor
cursor = conn.cursor()

In [ ]:
# Query all employees
query = "SELECT COUNT(*) FROM employees;"
cursor.execute(query)
print("Count of Employees: ", cursor.fetchall())

In [ ]:
#Count missing values per column
query ="""select 
    sum(case when employee_id is null then 1 else 0 end) as null_id,
    sum(case when name is null then 1 else 0 end) as null_name,
    sum(case when age is null then 1 else 0 end) as null_age,
    sum(case when salary is null then 1 else 0 end) as null_salary,
    sum(case when manager_id is null then 1 else 0 end) as null_mid,
    sum(case when department is null then 1 else 0 end) as null_department
from employees;"""
cursor.execute(query)
print("Count of Missing Values per Column: ", cursor.fetchall())

In [ ]:
#compute min max avg salary
query = "select min(salary) as Minimum, max(salary) as Maximum, avg(salary) as Average from employees;"
cursor.execute(query)
cursor.fetchall()

In [ ]:
#avg age per department
query = "select department, avg(age) from employees group by department;"
cursor.execute(query)
cursor.fetchall()

In [ ]:
#detect invalid salary
query = "select salary from employees where salary < 15000 and salary > 300000;"
cursor.execute(query)
cursor.fetchall()

In [ ]:
#detect invalid age
query = "select age from employees where age < 18 and age > 65;"
cursor.execute(query)
cursor.fetchall()

In [ ]:
#detect missing manager ids
query = "select manager_id from employees where manager_id is null;"
cursor.execute(query)
cursor.fetchall()

In [ ]:
# Detect duplicates
query = """
select name, age, salary, department, manager_id, count(*) as dup_count
from employees
group by name, age, salary, department, manager_id
having dup_count > 1;
"""
cursor.execute(query)
duplicates = cursor.fetchall()

print("=== Duplicate Records ===")
if not duplicates:
    print("No duplicates found.")
else:
    for row in duplicates:
        print(row)


In [ ]:
#average salary per department
query = "select department, avg(salary) from employees group by department;"

# --- Before Index ---
start = time.time()
cursor.execute(query)
rows_before = cursor.fetchall()
end = time.time()
time_before = end - start

print("Results before indexing:")
for row in rows_before:
    print(row)
print(f"Execution time (before index): {time_before:.6f} seconds\n")

cursor.execute("create index idx_department on employees(department);")

# --- After Index ---
start = time.time()
cursor.execute(query)
rows_after = cursor.fetchall()
end = time.time()
time_after = end - start

print("Results after indexing:")
for row in rows_after:
    print(row)
print(f"Execution time (after index): {time_after:.6f} seconds\n")

# Comparison
print("=== Performance Comparison ===")
print(f"Before index: {time_before:.6f} seconds")
print(f"After index : {time_after:.6f} seconds")


In [ ]:

print("=== Snapshot & Replication ===")

#snapshot table
cursor.execute("drop table if exists employees_snapshot;")
cursor.execute("create table employees_snapshot as select * from employees;")

cursor.execute("select count(*) from employees;")
original_count = cursor.fetchone()[0]
cursor.execute("select count(*) from employees_snapshot;")
snapshot_count = cursor.fetchone()[0]

print(f"Original employees count: {original_count}")
print(f"Snapshot employees count: {snapshot_count}")
print("Snapshot match:", "Yes" if original_count == snapshot_count else "No")

#replica table
cursor.execute("drop table if exists employees_replica;")
cursor.execute("create table employees_replica as select * from employees;")

cursor.execute("select count(*) from employees_replica;")
replica_count = cursor.fetchone()[0]
print(f"Replica employees count: {replica_count}")
print("Replica match:", "Yes" if original_count == replica_count else "No")

#simulate mismatch by deleting one record from replica
cursor.execute("delete from employees_replica limit 1;")
conn.commit()

cursor.execute("select count(*) from employees_replica;")
replica_count_after = cursor.fetchone()[0]
print(f"Replica employees count after deletion: {replica_count_after}")
print("Replica match after deletion:", "Yes" if original_count == replica_count_after else "No")



In [ ]:
print("=== Extraction Tasks ===\n")

# --- Periodical extraction ---
print("Periodical: Extract HR employees every 10 seconds for 1 minute\n")
start_time = time.time()
while time.time() - start_time < 60:
    cursor.execute("select * from employees where department = 'HR';")
    rows = cursor.fetchall()
    print(f"--- Extraction at {time.strftime('%H:%M:%S')} ---")
    for row in rows:
        print(row)
    print()
    time.sleep(10)

# --- On-request extraction ---
print("\nOn-request: Extract employees with salary > 100k\n")
cursor.execute("select * from employees where salary > 100000;")
rows = cursor.fetchall()
for row in rows:
    print(row)

# --- Event-driven extraction ---
print("\nEvent-driven: Extract when 3 new rows are inserted\n")

# Get current count
cursor.execute("select count(*) as cnt from employees;")
initial_count = cursor.fetchone()["cnt"]

print("Waiting for 3 new rows...")
while True:
    cursor.execute("select count(*) as cnt from employees;")
    new_count = cursor.fetchone()["cnt"]
    if new_count - initial_count >= 3:
        cursor.execute("select * from employees order by emp_id desc limit 3;")
        rows = cursor.fetchall()
        print("3 new rows detected:")
        for row in rows:
            print(row)
        break
    time.sleep(5)

# --- Immediate extraction ---
print("\nImmediate: Insert 1 new row and instantly extract it into staging\n")

# Create staging table if not exists
cursor.execute("""
    create table if not exists employees_staging as
    select * from employees where 1=0;
""")

# Insert 1 new row
cursor.execute("""
    insert into employees (name, age, salary, department, manager_id)
    values ('TempUser', 26, 50000, 'HR', 1);
""")
conn.commit()

# Extract that row and insert into staging
cursor.execute("select * from employees where name = 'TempUser' order by emp_id desc limit 1;")
new_row = cursor.fetchone()
if new_row:
    placeholders = ", ".join(["%s"] * len(new_row))
    cursor.execute(
        f"insert into employees_staging values ({placeholders});",
        tuple(new_row.values())
    )
    conn.commit()
    print("New row inserted into staging:")
    print(new_row)

In [ ]:
results = []

# Employees with highest salary per department (subquery)
cursor.execute("""
    select e.*
    from employees e
    where salary = (
        select max(salary)
        from employees
        where department = e.department
    );
""")
rows = cursor.fetchall()
results.append("=== Highest Salary per Department ===")
for row in rows:
    results.append(str(row))

# Employees with the same name but different departments
cursor.execute("""
    select name, group_concat(distinct department) as departments
    from employees
    group by name
    having count(distinct department) > 1;
""")
rows = cursor.fetchall()
results.append("\n=== Employees with Same Name but Different Departments ===")
for row in rows:
    results.append(str(row))

# Compare snapshot vs original table (added/removed rows)
results.append("\n=== Snapshot vs Original Differences ===")
cursor.execute("""
    select * from employees
    where emp_id not in (select emp_id from employees_snapshot);
""")
added = cursor.fetchall()
results.append("Added in employees (not in snapshot):")
for row in added:
    results.append(str(row))

cursor.execute("""
    select * from employees_snapshot
    where emp_id not in (select emp_id from employees);
""")
removed = cursor.fetchall()
results.append("Removed from employees (still in snapshot):")
for row in removed:
    results.append(str(row))

# Detect salary outliers using AVG + STDDEV
cursor.execute("select avg(salary) as avg_sal, stddev(salary) as std_sal from employees;")
stats = cursor.fetchone()
avg_sal = stats["avg_sal"]
std_sal = stats["std_sal"]

cursor.execute("""
    select * from employees
    where salary > %s + 2 * %s or salary < %s - 2 * %s;
""", (avg_sal, std_sal, avg_sal, std_sal))
rows = cursor.fetchall()
results.append("\n=== Salary Outliers (±2 STDDEV) ===")
for row in rows:
    results.append(str(row))

# Department with youngest average age
cursor.execute("""
    select department, avg(age) as avg_age
    from employees
    group by department
    order by avg_age asc
    limit 1;
""")
row = cursor.fetchone()
results.append("\n=== Department with Youngest Average Age ===")
results.append(str(row))

# Save results into file
with open("dwl_lab_4.txt", "w") as f:
    f.write("\n".join(results))

print("All results saved into dwl_lab_4.txt")

In [ ]:
cursor.close()
conn.close()